In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path().resolve().parents[0])) 

import pandas as pd

from config import RAW_DATA_DIR, PROCESSED_DATA_DIR

In [2]:
reviews_df = pd.read_csv(RAW_DATA_DIR / "Amazon_Reviews.csv")

In [3]:
# Get general structure of dataset
print(f"{reviews_df.shape[0]} rows, {reviews_df.shape[1]} columns")

21214 rows, 9 columns


In [4]:
# Get list of column names
print(reviews_df.columns)

Index(['Reviewer Name', 'Profile Link', 'Country', 'Review Count',
       'Review Date', 'Rating', 'Review Title', 'Review Text',
       'Date of Experience'],
      dtype='str')


In [5]:
# Remove columns that do not meaningfully help predict rating
reviews_df = reviews_df.drop(columns=["Reviewer Name", "Profile Link"])

# Verify columns are dropped 
print(reviews_df.columns)

Index(['Country', 'Review Count', 'Review Date', 'Rating', 'Review Title',
       'Review Text', 'Date of Experience'],
      dtype='str')


In [6]:
# Check duplicate rows before further cleaning
print("Exact duplicates:", reviews_df.duplicated().sum())

content_dupes = reviews_df.duplicated(
    subset=["Review Date", "Rating", "Review Title", "Review Text", "Date of Experience"]
).sum()
print("Content-level duplicates:", content_dupes)

# Remove exact duplicates, then duplicate review entries based on core review content
reviews_df = reviews_df.drop_duplicates()
reviews_df = reviews_df.drop_duplicates(
    subset=["Review Date", "Rating", "Review Title", "Review Text", "Date of Experience"]
)

print("Rows after duplicate removal:", len(reviews_df))

Exact duplicates: 158
Content-level duplicates: 158
Rows after duplicate removal: 21056


In [7]:
# Check missing values
print(reviews_df.isna().sum())

Country                 2
Review Count            1
Review Date             1
Rating                  1
Review Title            1
Review Text             1
Date of Experience    109
dtype: int64


In [8]:
# Drop NA ratings 
reviews_df = reviews_df.dropna(subset=["Rating"])

In [9]:
# Treat missing text-related features as empty
reviews_df["Review Title"] = reviews_df["Review Title"].fillna("")
reviews_df["Review Text"] = reviews_df["Review Text"].fillna("")

In [10]:
# Replace missing Country values with 'Unknown' 
reviews_df["Country"] = reviews_df["Country"].fillna("Unknown")

In [11]:
# Convert Review Count column into a numerical value
reviews_df["Review Count"] = (
    reviews_df["Review Count"]
    .str.strip()
    .str.split(" ")
    .str[0]
)

reviews_df["Review Count"] = pd.to_numeric(
    reviews_df["Review Count"], errors="coerce"
)

# Impute missing Review Count values with median value, then cast to int
reviews_df["Review Count"] = reviews_df["Review Count"].fillna(
    reviews_df["Review Count"].median()
).astype(int)

# Verify that Review Count is successfully converted to int
assert reviews_df["Review Count"].dtype == int

In [12]:
# Drop missing Review Date columns
reviews_df = reviews_df.dropna(subset=["Review Date"])

# For missing Date of Experience values assume it is equivalent to the review date
reviews_df["Date of Experience"] = reviews_df["Date of Experience"].fillna(reviews_df["Review Date"])

In [13]:
# Convert date-related columns to datetime type
# Handle malformed dates in Date of Experience
reviews_df["Review Date"] = pd.to_datetime(
    reviews_df["Review Date"],
    errors="coerce",
    format="mixed",
    utc=True
)

reviews_df["Date of Experience"] = pd.to_datetime(
    reviews_df["Date of Experience"],
    errors="coerce",
    format="mixed",
    utc=True
)

In [14]:
# Strip whitespace on text-related columns 
reviews_df["Review Text"] = reviews_df["Review Text"].str.strip()
reviews_df["Review Title"] = reviews_df["Review Title"].str.strip()
reviews_df["Country"] = reviews_df["Country"].str.strip().str.upper()

# Normalize spacing and unicode in review text
reviews_df["Review Text"] = reviews_df["Review Text"].str.replace(r"\s+", " ", regex=True)
reviews_df["Review Text"] = reviews_df["Review Text"].str.replace("\u2028", " ")
reviews_df["Review Text"] = reviews_df["Review Text"].str.replace("\u2029", " ")

In [15]:
reviews_df["Rating"] = reviews_df["Rating"].str.split(" ").str[1].astype(int)
print(reviews_df["Rating"].dtype)
print(reviews_df["Rating"].unique())

int64
[1 5 2 4 3]


In [16]:
# Check that there are no more missing values present
print(reviews_df.isna().sum())

Country               0
Review Count          0
Review Date           0
Rating                0
Review Title          0
Review Text           0
Date of Experience    0
dtype: int64


In [17]:
# Rename columns
reviews_df.columns = (
    reviews_df.columns.str.strip().str.lower().str.replace(" ", "_")
)

# Verfy columns renamed properly
print(reviews_df.columns)

Index(['country', 'review_count', 'review_date', 'rating', 'review_title',
       'review_text', 'date_of_experience'],
      dtype='str')


In [18]:
# Final check of column dtypes
print(reviews_df.dtypes)

country                               str
review_count                        int64
review_date           datetime64[us, UTC]
rating                              int64
review_title                          str
review_text                           str
date_of_experience    datetime64[us, UTC]
dtype: object


In [19]:
# Save cleaned dataset to Parquet file
reviews_df.to_parquet(PROCESSED_DATA_DIR / "amazon_reviews_clean.parquet", index=False)